In [1]:
%pip install openai

  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.12.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.1 MB/s  0:00:00
Using cached anyio-4.12.0-py3-none-any.whl (113 kB

In [3]:
import os
from getpass import getpass
from openai import OpenAI
import duckdb

In [ ]:
OPENAI_API_KEY = getpass("Enter your OpenAI API key:")

In [ ]:
#instantiate OpenAI client
client = OpenAI(
    api_key = OPENAI_API_KEY
)

In [ ]:
results = []

for i, text in enumerate():
    print(f"--- Processing File {i+1} ---")
    response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f":\n\n{text}"}
      ],
      temperature = 0, # controls randomness of output
    )
    result = response.choices[0].message.content
    results.append(result)
    print(result)
    print("\n" + "="*50 + "\n")

In [ ]:
# Persist the DB file so you don’t regenerate data every time
con = duckdb.connect("tpcds.duckdb")

In [5]:
con.execute("INSTALL tpcds")
con.execute("LOAD tpcds")
print("TPC-DS extension loaded.")

TPC-DS extension loaded.


In [6]:
SCALE_FACTOR = 1

con.execute(f"CALL dsdgen(sf={SCALE_FACTOR})")
print("TPC-DS data generated.")

TPC-DS data generated.


In [11]:
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    ORDER BY table_name
""").df()

tables

,table_name
0,call_center
1,catalog_page
2,catalog_returns
3,catalog_sales
4,customer
5,customer_address
6,customer_demographics
7,date_dim
8,household_demographics
9,income_band


In [ ]:
df_q1 = con.execute("""
    SELECT *
    FROM tpcds_queries()
    WHERE query_nr = 1
""").df()

print(df_q1[0])

KeyError: 0

In [34]:
for i in range(99):
    results = con.execute(f"PRAGMA tpcds({i+1});").df()
    print(results)

       c_customer_id
0   AAAAAAAAAAABBAAA
1   AAAAAAAAAAADBAAA
2   AAAAAAAAAAADBAAA
3   AAAAAAAAAACJAAAA
4   AAAAAAAAAACNAAAA
..               ...
95  AAAAAAAAACEBBAAA
96  AAAAAAAAACECBAAA
97  AAAAAAAAACECBAAA
98  AAAAAAAAACFAAAAA
99  AAAAAAAAACFABAAA

[100 rows x 1 columns]
      d_week_seq1    r1    r2    r3    r4    r5    r6  \
0            5270  3.52  2.20  1.74  1.60  3.01  3.89   
1            5270  3.52  2.20  1.74  1.60  3.01  3.89   
2            5270  3.52  2.20  1.74  1.60  3.01  3.89   
3            5270  3.52  2.20  1.74  1.60  3.01  3.89   
4            5270  3.52  2.20  1.74  1.60  3.01  3.89   
...           ...   ...   ...   ...   ...   ...   ...   
2508         5322  4.67  4.67  0.98  1.99  1.74  5.66   
2509         5322  4.67  4.67  0.98  1.99  1.74  5.66   
2510         5322  4.67  4.67  0.98  1.99  1.74  5.66   
2511         5322  4.67  4.67  0.98  1.99  1.74  5.66   
2512         5322  4.67  4.67  0.98  1.99  1.74  5.66   

      round((sat_sales1 / sat_sales2), 